# Homework 03: The Triphasic Reach — Why Muscles Need Damping

**Computational Sensorimotor Control** | Week 3 | Module 1: The Biological Plant

**Due:** One week from assignment | **Total:** 100 points

---

## The Experiment

You want to reach 5–8 cm and *stop*. This requires three phases of muscle activation — the **triphasic pattern** first described by Wachholder (1928) and extensively studied since (Hallett et al. 1975; Berardelli et al. 1996):

1. **Agonist burst (AG1):** Accelerate the limb toward the target
2. **Antagonist burst (ANT):** Brake the limb before it overshoots
3. **Agonist correction (AG2) + co-contraction:** Fine-tune and stabilize

In this homework, you will design a triphasic activation pattern and test it on **two versions** of the arm model:

- **Model A:** Our Gribble muscle model + joint viscous damping (B = 3.0 N·m·s/rad), representing passive tissue viscosity
- **Model B:** Our Gribble muscle model alone (B = 0) — no extra damping

You will discover that the same activation pattern produces clean reaching in Model A but fails to settle in Model B. This is the strongest motivation for the λ model (Week 4), which provides length-dependent stiffness and damping automatically.

**You will need:** Your `DynamicsEngine` class from Lab 03.

---
## Part 0: Setup (0 pts)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Setup complete.')

### Paste Your DynamicsEngine from Lab 03

Paste your complete code from Lab 03: `Arm`, `Muscle`, all supporting functions (`force_length`, `calcium_dynamics`, `force_velocity_multiplier`, `mass_matrix`, `coriolis`, `joint_accelerations`, `rk4_step`, `muscle_length`, `muscle_velocity`, `compute_torques`), and the `DynamicsEngine` class.

**Important:** Your `force_length` function should include the neural drive gain G = 20 mm from Lab 02.

In [ ]:
# PASTE your complete Lab 03 code here



### Modify `joint_accelerations` to Support Damping

To compare the two models, we need to add an optional **joint viscous damping** term B·dq/dt to the equations of motion:

d²q/dt² = M(q)⁻¹ [τ − C(q,dq/dt)·dq/dt − B·dq/dt]

When B = 0, this is the original Gribble model. When B > 0, it adds passive tissue viscosity at each joint.

COMPLETE the modification below.

In [ ]:
# TODO: COMPLETE — modify joint_accelerations to accept B parameter
def joint_accelerations(q, qdot, tau, B=0.0):
    """Compute joint accelerations with optional viscous damping.
    
    Parameters:
        q: joint angles [theta1, theta2]
        qdot: joint velocities [dtheta1/dt, dtheta2/dt]
        tau: joint torques [tau1, tau2]
        B: viscous damping coefficient (N·m·s/rad), default 0
    Returns:
        qddot: joint accelerations
    """
    M = mass_matrix(q)
    c = coriolis(q, qdot)
    damping = B * qdot  # viscous damping at each joint
    
    return np.linalg.solve(M, tau - c - damping)

# Verify: with B=0, should match Lab 03
q_test = np.array([np.radians(55), np.radians(75)])
acc_0 = joint_accelerations(q_test, np.zeros(2), np.array([1.0, 0.0]), B=0.0)
acc_3 = joint_accelerations(q_test, np.array([1.0, 0.0]), np.array([1.0, 0.0]), B=3.0)
print(f"B=0, zero velocity: d²θ/dt² = [{acc_0[0]:.3f}, {acc_0[1]:.3f}]")
print(f"B=3, unit velocity:  d²θ/dt² = [{acc_3[0]:.3f}, {acc_3[1]:.3f}]")
print(f"Damping reduces acceleration ✓" if acc_3[0] < acc_0[0] else "ERROR")

### Create Two DynamicsEngine Instances

COMPLETE the cell below to create Model A (damped) and Model B (undamped). You will need to modify the DynamicsEngine's `_kinematic_derivatives` method to pass B through, or create a wrapper.

In [ ]:
# Create the 6 muscles (same parameters as Lab 03)
pec = Muscle('pectoralis', 14.9, 258.5, 0.04, 0.0, 0.26)
bic_long = Muscle('biceps_long', 11.0, 150.0, 0.0, 0.03, 0.26)
bic_short = Muscle('biceps_short', 2.1, 100.0, 0.025, 0.03, 0.29)
deltoid = Muscle('deltoid', 14.9, 258.5, -0.04, 0.0, 0.26)
tri_lat = Muscle('triceps_lat', 12.1, 200.0, 0.0, -0.02, 0.26)
tri_long = Muscle('triceps_long', 6.7, 100.0, -0.04, -0.02, 0.32)

muscles_A = [Muscle('pec',14.9,258.5,0.04,0.0,0.26), Muscle('bic_l',11.0,150.0,0.0,0.03,0.26),
             Muscle('bic_s',2.1,100.0,0.025,0.03,0.29), Muscle('delt',14.9,258.5,-0.04,0.0,0.26),
             Muscle('tri_l',12.1,200.0,0.0,-0.02,0.26), Muscle('tri_lg',6.7,100.0,-0.04,-0.02,0.32)]
muscles_B = [Muscle('pec',14.9,258.5,0.04,0.0,0.26), Muscle('bic_l',11.0,150.0,0.0,0.03,0.26),
             Muscle('bic_s',2.1,100.0,0.025,0.03,0.29), Muscle('delt',14.9,258.5,-0.04,0.0,0.26),
             Muscle('tri_l',12.1,200.0,0.0,-0.02,0.26), Muscle('tri_lg',6.7,100.0,-0.04,-0.02,0.32)]

arm = Arm()

# TODO: COMPLETE — create two simulate functions (or modify DynamicsEngine)
# Model A: B = 3.0 N·m·s/rad (damped)
# Model B: B = 0 (undamped, original Gribble)

def simulate(act_fn, muscles, T=1.0, dt=0.0001, B=0.0):
    """Simulate arm movement with given activation function and damping."""
    for m in muscles:
        m.reset()
    q_ref = np.array([np.radians(55), np.radians(75)])
    state = np.concatenate([q_ref, np.zeros(2)])
    
    t = np.arange(0, T, dt)
    states = np.zeros((len(t), 4))
    hand = np.zeros((len(t), 2))
    
    for i in range(len(t)):
        states[i] = state
        hand[i] = arm.forward_kinematics(state[:2])
        q, qdot = state[:2], state[2:]
        
        # Phase 1: muscles update once
        activations = act_fn(t[i])
        tau = compute_torques(muscles, q, qdot, activations, dt)
        
        # Phase 2: RK4 on kinematics with fixed tau and damping B
        deriv = lambda s: np.array([s[2], s[3], 
                                     *joint_accelerations(s[:2], s[2:], tau, B)])
        state = rk4_step(state, deriv, dt)
    
    return t, states, hand

# Quick test
t_test, s_test, h_test = simulate(lambda t: np.zeros(6), muscles_A, T=0.01, B=3.0)
print(f"Simulate works. Start: θ₁={np.degrees(s_test[0,0]):.1f}°, θ₂={np.degrees(s_test[0,1]):.1f}°")
start_hand = arm.forward_kinematics(np.array([np.radians(55), np.radians(75)]))
print(f"Start hand: ({start_hand[0]*100:.1f}, {start_hand[1]*100:.1f}) cm")

---
## Part 1: Agonist Only — What Happens Without Braking? (15 pts)

### Task 1.1 — Design an Agonist Activation Profile (5 pts)

COMPLETE the activation function below. Activate the pectoralis (muscle 0) at a = 1.0 and biceps short (muscle 2) at a = 0.4 from t = 20 ms to t = 200 ms. All other muscles at zero.

In [ ]:
# TODO: COMPLETE — define the agonist-only activation
def agonist_only(t):
    a = np.zeros(6)
    if 0.02 <= t < 0.20:
        a[0] = ...  # pectoralis
        a[2] = ...  # biceps short
    return a

### Task 1.2 — Simulate on Both Models (5 pts)

COMPLETE the cell below to simulate the agonist-only activation on both Model A and Model B for 1 second. Plot hand displacement vs. time for both on the same axes.

In [ ]:
# TODO: COMPLETE — simulate both models
t_A, s_A, h_A = simulate(agonist_only, muscles_A, T=1.0, B=3.0)
t_B, s_B, h_B = simulate(agonist_only, muscles_B, T=1.0, B=0.0)

# Compute hand displacement from start
disp_A = np.linalg.norm(h_A - start_hand, axis=1) * 100  # cm
disp_B = np.linalg.norm(h_B - start_hand, axis=1) * 100

# TODO: COMPLETE — plot
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_A*1000, disp_A, '#2E86AB', lw=2.5, label='Model A (B=3.0, damped)')
ax.plot(t_B*1000, disp_B, '#E74C3C', lw=2.5, label='Model B (B=0, undamped)')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Hand displacement (cm)')
ax.set_title('Agonist Only: No Braking')
ax.legend(); plt.show()

print(f"Model A at 500ms: {disp_A[5000]:.1f} cm, vel ≈ ... cm/s")
print(f"Model B at 500ms: {disp_B[5000]:.1f} cm, vel ≈ ... cm/s")

### Question 1.1 (5 pts)

**(a)** In Model A, the hand decelerates and nearly stops even without antagonist braking. Why? What physical mechanism provides the damping?

**(b)** In Model B, the hand keeps moving indefinitely. Why doesn't the force-velocity relationship provide enough damping to stop it?

*Your answer here:*


---
## Part 2: Design the Triphasic Reach — Model A (25 pts)

Now design a three-phase activation pattern that moves the hand ~5 cm and stops. Work with Model A first (B = 3.0) where the triphasic pattern works.

### Task 2.1 — Add Antagonist Braking (10 pts)

Start with your agonist burst from Part 1, then add an antagonist burst (deltoid + triceps long) that overlaps with the end of the agonist. The antagonist should start **before** the agonist ends to account for the calcium delay.

COMPLETE the activation function below. Suggested starting values are provided — you will tune them in Task 2.3.

In [ ]:
def agonist_antagonist(t):
    a = np.zeros(6)
    # AG1: agonist burst
    if 0.02 <= t < 0.20:
        a[0] = 1.0   # pectoralis
        a[2] = 0.4   # biceps short
    # ANT: antagonist brake (overlapping!)
    if 0.18 <= t < 0.30:
        a[3] = ...   # TODO: COMPLETE — deltoid (try 0.8)
        a[5] = ...   # TODO: COMPLETE — triceps long (try 0.3)
    return a

# Simulate on Model A
t_ag, s_ag, h_ag = simulate(agonist_antagonist, muscles_A, T=1.0, B=3.0)
disp_ag = np.linalg.norm(h_ag - start_hand, axis=1) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_A*1000, disp_A, '#2E86AB', lw=1.5, ls='--', alpha=0.5, label='Agonist only')
ax.plot(t_ag*1000, disp_ag, '#E67E22', lw=2.5, label='Agonist + Antagonist')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Hand displacement (cm)')
ax.set_title('Model A: Adding Antagonist Braking')
ax.legend(); plt.show()

print(f"Peak displacement: {np.max(disp_ag):.1f} cm at {t_ag[np.argmax(disp_ag)]*1000:.0f} ms")
print(f"Displacement at 500ms: {disp_ag[5000]:.1f} cm")

### Task 2.2 — Add Co-Contraction Hold (5 pts)

After the antagonist burst, add a sustained low-level co-contraction (both agonist and antagonist at equal activation) to stabilize the arm at the new position.

COMPLETE the full triphasic activation function.

In [ ]:
def triphasic(t):
    a = np.zeros(6)
    # AG1: agonist burst (accelerate)
    if 0.02 <= t < 0.20:
        a[0] = 1.0   # pectoralis
        a[2] = 0.4   # biceps short
    # ANT: antagonist brake (decelerate)
    if 0.18 <= t < 0.30:
        a[3] = 0.8   # deltoid
        a[5] = 0.3   # triceps long
    # AG2 + hold: symmetric co-contraction (stabilize)
    if t >= 0.28:
        a[0] = ...   # TODO: COMPLETE — tonic pec (try 0.25)
        a[3] = ...   # TODO: COMPLETE — tonic delt (try 0.25)
    return a

# Simulate on Model A
t_tri, s_tri, h_tri = simulate(triphasic, muscles_A, T=1.0, B=3.0)
disp_tri = np.linalg.norm(h_tri - start_hand, axis=1) * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_A*1000, disp_A, '#2E86AB', lw=1.5, ls='--', alpha=0.5, label='Agonist only')
ax.plot(t_ag*1000, disp_ag, '#E67E22', lw=1.5, ls='--', alpha=0.5, label='AG + ANT')
ax.plot(t_tri*1000, disp_tri, '#27AE60', lw=2.5, label='Full triphasic')
ax.set_xlabel('Time (ms)'); ax.set_ylabel('Hand displacement (cm)')
ax.set_title('Model A: Full Triphasic Pattern')
ax.legend(); plt.show()

### Task 2.3 — Tune the Pattern (5 pts)

The suggested activation levels above are a starting point. COMPLETE the cell below to tune your triphasic pattern so that:
1. The hand moves at least 5 cm
2. The hand velocity drops below 1 cm/s within 800 ms
3. The final position is stable (not drifting)

Report your final activation levels and timing.

In [ ]:
# TODO: COMPLETE — copy your triphasic function here with tuned parameters
# Experiment with:
#   - AG1 duration and amplitude
#   - ANT onset time (earlier = more anticipatory) and amplitude
#   - Hold activation level

def triphasic_tuned(t):
    a = np.zeros(6)
    # Your tuned parameters:
    if 0.02 <= t < ...:     a[0] = ...; a[2] = ...    # AG1
    if ... <= t < ...:       a[3] = ...; a[5] = ...    # ANT
    if t >= ...:             a[0] = ...; a[3] = ...    # Hold
    return a

# Simulate
t_tuned, s_tuned, h_tuned = simulate(triphasic_tuned, muscles_A, T=1.0, B=3.0)
disp_tuned = np.linalg.norm(h_tuned - start_hand, axis=1) * 100
vel_tuned = np.zeros(len(t_tuned))
vel_tuned[1:] = np.linalg.norm(np.diff(h_tuned, axis=0) / 0.0001, axis=1) * 100  # cm/s

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
ax1.plot(t_tuned*1000, disp_tuned, '#27AE60', lw=2.5)
ax1.set_ylabel('Displacement (cm)'); ax1.set_title('Model A: Tuned Triphasic Reach')
ax2.plot(t_tuned*1000, vel_tuned, '#E74C3C', lw=2)
ax2.axhline(1.0, color='gray', ls=':', alpha=0.5, label='1 cm/s threshold')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Hand velocity (cm/s)')
ax2.legend(); plt.tight_layout(); plt.show()

# Check criteria
peak_disp = np.max(disp_tuned)
final_disp = disp_tuned[-1]
final_vel = vel_tuned[-1]
print(f"Peak displacement: {peak_disp:.1f} cm")
print(f"Final displacement: {final_disp:.1f} cm")
print(f"Final velocity: {final_vel:.1f} cm/s")
print(f"Criteria: ≥5 cm {'✓' if peak_disp>=5 else '✗'}, "
      f"vel<1 cm/s {'✓' if final_vel<1 else '✗'}, "
      f"stable {'✓' if abs(final_vel)<1 else '✗'}")

### Question 2.1 (5 pts)

**(a)** Why must the antagonist burst start *before* the agonist ends? What happens if you wait until the agonist is fully off before starting the antagonist?

**(b)** What role does the co-contraction hold phase play? What happens if you omit it?

*Your answer here:*


---
## Part 3: The Same Pattern on Model B — It Fails (20 pts)

### Task 3.1 — Apply Your Tuned Triphasic to Model B (5 pts)

Take the **exact same** activation function you tuned in Part 2 and run it on Model B (B = 0). COMPLETE the cell below.

In [ ]:
# Simulate the SAME triphasic_tuned on Model B (no damping)
t_B_tri, s_B_tri, h_B_tri = simulate(triphasic_tuned, muscles_B, T=1.0, B=0.0)
disp_B_tri = np.linalg.norm(h_B_tri - start_hand, axis=1) * 100
vel_B_tri = np.zeros(len(t_B_tri))
vel_B_tri[1:] = np.linalg.norm(np.diff(h_B_tri, axis=0) / 0.0001, axis=1) * 100

### Task 3.2 — Side-by-Side Comparison (10 pts)

COMPLETE the cell below to plot Model A and Model B trajectories on the same figure (two panels: displacement and velocity).

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Displacement
ax1.plot(t_tuned*1000, disp_tuned, '#2E86AB', lw=2.5, label='Model A (B=3.0, damped)')
ax1.plot(t_B_tri*1000, disp_B_tri, '#E74C3C', lw=2.5, label='Model B (B=0, undamped)')
ax1.set_ylabel('Hand displacement (cm)')
ax1.set_title('Same Triphasic Pattern, Two Models')
ax1.legend()

# Velocity
ax2.plot(t_tuned*1000, vel_tuned, '#2E86AB', lw=2, label='Model A')
ax2.plot(t_B_tri*1000, vel_B_tri, '#E74C3C', lw=2, label='Model B')
ax2.axhline(1.0, color='gray', ls=':', alpha=0.5, label='1 cm/s threshold')
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Hand velocity (cm/s)')
ax2.legend()

plt.tight_layout(); plt.show()

print(f"Model A — final disp: {disp_tuned[-1]:.1f} cm, final vel: {vel_tuned[-1]:.1f} cm/s")
print(f"Model B — final disp: {disp_B_tri[-1]:.1f} cm, final vel: {vel_B_tri[-1]:.1f} cm/s")

### Question 3.1 (5 pts)

**(a)** Describe the difference between the two trajectories. Does Model B overshoot? Does it settle?

**(b)** The antagonist braking reduces Model B's velocity temporarily, but the arm starts drifting again. Why? What is missing from Model B that Model A has?

*Your answer here:*


---
## Part 4: Can Co-Contraction Save Model B? (15 pts)

### Task 4.1 — Increase Co-Contraction (10 pts)

One way to add damping to Model B is to increase the tonic co-contraction level during the hold phase. This increases the force-velocity damping from both muscles.

COMPLETE the cell below to test four co-contraction levels (0.1, 0.3, 0.5, and 0.8) on Model B with the same triphasic timing. Plot all three on one figure.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

# Model A reference
ax1.plot(t_tuned*1000, disp_tuned, '#2E86AB', lw=2, ls='--', alpha=0.5, label='Model A (reference)')

for cocon, color, ls in [(0.1, '#E74C3C', '-'), (0.3, '#E67E22', '-'), (0.5, '#9B59B6', '-'), (0.8, '#27AE60', '-')]:
    # TODO: COMPLETE — create triphasic with different hold levels
    def triphasic_cocon(t, cc=cocon):
        a = np.zeros(6)
        # Use your tuned AG1 and ANT timing from Part 2
        if 0.02 <= t < 0.20: a[0] = 1.0; a[2] = 0.4
        if 0.18 <= t < 0.30: a[3] = 0.8; a[5] = 0.3
        if t >= 0.28: a[0] = cc; a[3] = cc    # symmetric co-contraction
        return a
    
    t_cc, s_cc, h_cc = simulate(triphasic_cocon, muscles_B, T=1.0, B=0.0)
    disp_cc = np.linalg.norm(h_cc - start_hand, axis=1) * 100
    vel_cc = np.zeros(len(t_cc))
    vel_cc[1:] = np.linalg.norm(np.diff(h_cc, axis=0) / 0.0001, axis=1) * 100
    
    ax1.plot(t_cc*1000, disp_cc, color=color, lw=2, ls=ls, label=f'Model B, hold={cocon}')
    ax2.plot(t_cc*1000, vel_cc, color=color, lw=2, label=f'hold={cocon}')
    
    print(f"Co-contraction={cocon}: final disp={disp_cc[-1]:.1f} cm, final vel={vel_cc[-1]:.1f} cm/s")

ax1.set_ylabel('Hand displacement (cm)'); ax1.set_title('Model B: Can Co-Contraction Fix the Drift?')
ax1.legend(fontsize=9)
ax2.axhline(1.0, color='gray', ls=':', alpha=0.5)
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Hand velocity (cm/s)'); ax2.legend(fontsize=9)
plt.tight_layout(); plt.show()

### Question 4.1 (5 pts)

**(a)** Does increasing co-contraction help Model B settle? At what level?

**(b)** What is the metabolic cost of this strategy? Both muscles are active but producing opposing torques — where does the energy go?

**(c)** In Week 4, the λ model will provide length-dependent activation: A = [l − λ]⁺. This means the muscle automatically activates more when stretched and less when shortened. How would this create both stiffness *and* damping without the metabolic cost of co-contraction?

**(d)** Does a = 0.8 actually stabilize Model B? Compare its settling behavior and total isometric force to Model A.


*Your answer here:*


---
## Part 5: Comparison to HW02 Prediction + Write-Up (25 pts)

### Task 5.1 — Compute HW02-Style Overshoot (10 pts)

In HW02, you predicted overshoot as: overshoot = peak hand velocity × calcium delay (58 ms).

COMPLETE the cell below to:
1. Measure the peak hand velocity from your agonist-only simulation (Part 1, Model A)
2. Compute the HW02 prediction
3. Compare to the actual peak displacement from the full dynamics simulation

In [ ]:
# Hand velocity from agonist-only, Model A
vel_A_ag = np.zeros(len(t_A))
vel_A_ag[1:] = np.linalg.norm(np.diff(h_A, axis=0) / 0.0001, axis=1)

# TODO: COMPLETE — compute peak velocity and HW02 prediction
peak_hand_vel = np.max(vel_A_ag)  # m/s
calcium_delay = 0.058  # seconds
hw02_prediction = peak_hand_vel * calcium_delay * 100  # cm

print(f"Peak hand velocity (agonist, Model A): {peak_hand_vel*100:.1f} cm/s = {peak_hand_vel:.2f} m/s")
print(f"HW02 predicted overshoot (v × 58ms): {hw02_prediction:.1f} cm")
print(f"Actual peak displacement (agonist, Model A): {np.max(disp_A):.1f} cm")
print(f"Actual peak displacement (triphasic, Model A): {np.max(disp_tuned):.1f} cm")
print(f"\nDifference: HW02 prediction vs actual: {abs(hw02_prediction - np.max(disp_A)):.1f} cm")

### Question 5.1 (included in write-up)

Does the HW02 simplified prediction match the full dynamics simulation? If not, explain what additional factors the full simulation includes that HW02 ignored (interaction torques, Coriolis forces, passive stiffness, force-velocity damping, configuration-dependent inertia).

### Task 5.2 — Methods & Predictions Write-Up (15 pts)

Write a **Methods and Results** section (400–600 words):

**Methods (~200 words):** Describe the two models (A and B), the triphasic activation pattern you designed (report your tuned parameters), and how you compared them.

**Results (~200–400 words):** Report the key findings:
1. Model A: the triphasic pattern produces a ~5 cm reach that settles within 800 ms
2. Model B: the same pattern fails to settle — the arm drifts because it lacks damping
3. Co-contraction partially rescues Model B but at metabolic cost
4. The HW02 simplified prediction differs from the full simulation by X cm because of [factors]
5. Conclude: the direct activation model lacks the intrinsic damping needed for stable reaching, motivating the λ model (Week 4)

**Include two figures:** (1) the Model A vs. Model B comparison from Task 3.2, (2) the co-contraction comparison from Task 4.1.

**Format:** Third person, past tense.

*Your write-up here:*


---
## Looking Ahead: The λ Model Solves This Problem

The core issue is that with direct activation a(t), the muscle has no way to "know" its own length. It produces force based solely on the commanded activation, not on what the joint is actually doing. The joint viscous damping B in Model A was a placeholder for the missing physics.

In **Week 4**, the λ model replaces direct activation with threshold control:

**A = [l − λ]⁺**

Now activation depends on muscle length l. When the muscle is stretched beyond threshold (l > λ), activation increases automatically. When it shortens (l < λ), activation drops to zero. This creates:
- **Length-dependent stiffness:** displaced muscles push back toward equilibrium
- **Velocity-dependent damping:** through the force-velocity relationship, faster stretches produce more resistive force
- **No metabolic cost at equilibrium:** when l = λ, activation is zero

The λ model doesn't need the B term. It doesn't need hand-tuned triphasic patterns. A simple shift of λ from one equilibrium to another produces bell-shaped velocity profiles, triphasic EMG patterns, and stable reaching — all emergent from the muscle mechanics.

---
## Summary

You designed a triphasic activation pattern (AG1 → ANT → AG2 + hold) and tested it on two models:

- **Model A (with damping):** Clean reaching — the arm accelerates, brakes, and settles at ~5 cm within 800 ms. Joint viscous damping (B = 3.0 N·m·s/rad) dissipates kinetic energy during deceleration.
- **Model B (without damping):** The same pattern fails — the arm oscillates and drifts because the Gribble muscle model with direct activation provides insufficient damping. The force-velocity relationship helps but is too weak when calcium-filtered activation is low.
- **Co-contraction partially rescues Model B** but at high metabolic cost.
- **HW02's simplified prediction** (v × 58 ms) differs from the full dynamics because it ignores interaction torques, Coriolis forces, and configuration-dependent inertia.

**Key takeaway:** Stable reaching requires either external damping (Model A) or length-dependent muscle activation (the λ model, Week 4). Direct activation alone cannot produce the intrinsic stiffness and damping needed to stabilize the arm at a new posture.